# Attention Variants

Core ML Task 2 notebook. This keeps the baseline structure, but makes attention mechanisms swappable and logs every experiment independently.

# Setup

In [ ]:
!pip install -q datasets transformers tqdm pandas

# Imports

In [ ]:
import math
import os
import random
import time
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import load_dataset
from transformers import AutoTokenizer
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# Drive Logging

In [ ]:
# In Colab, this mounts Google Drive and writes logs there.
# Outside Colab, logs are written to a local folder.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    LOG_DIR = "/content/drive/MyDrive/SAiDL_Assignment/core_ml_attention_variants"
except Exception:
    LOG_DIR = "attention_variant_logs"

os.makedirs(LOG_DIR, exist_ok=True)
print("Logging to:", LOG_DIR)

# Config

In [ ]:
class Config:
    vocab_size = 50257
    block_size = 1024
    n_layer = 4
    n_head = 4
    n_embd = 256
    dropout = 0.1
    learning_rate = 3e-4

    # Attention-variant settings
    window_size = 256      # for sliding-window/local attention
    kv_heads = 1           # 1 = MQA; >1 and <n_head = GQA
    linear_attention_eps = 1e-6


def make_config(block_size):
    return SimpleNamespace(
        vocab_size=Config.vocab_size,
        block_size=block_size,
        n_layer=Config.n_layer,
        n_head=Config.n_head,
        n_embd=Config.n_embd,
        dropout=Config.dropout,
        learning_rate=Config.learning_rate,
        window_size=Config.window_size,
        kv_heads=Config.kv_heads,
        linear_attention_eps=Config.linear_attention_eps,
    )


CONTEXT_LENGTHS = [512, 1024, 2048]
BATCH_SIZES = {
    512: 16,
    1024: 8,
    2048: 4,
}

# Screening setup. Increase after smoke tests succeed.
EXPERIMENT_EPOCHS = 15
SEED = 42

# Load Dataset + Tokenizer

In [ ]:
dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Tokenize Once

In [ ]:
def tokenize(example):
    return tokenizer(example["text"])


tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])
print(tokenized)

# Build Context-Length Datasets + DataLoaders

In [ ]:
lm_dataset_cache = {}
dataloader_cache = {}


def build_lm_dataset(block_size):
    if block_size in lm_dataset_cache:
        return lm_dataset_cache[block_size]

    def group_texts(examples):
        concatenated = sum(examples["input_ids"], [])
        total_length = (len(concatenated) // block_size) * block_size

        input_ids = [
            concatenated[i:i + block_size]
            for i in range(0, total_length, block_size)
        ]

        return {"input_ids": input_ids, "labels": input_ids.copy()}

    lm_datasets = tokenized.map(
        group_texts,
        batched=True,
        remove_columns=tokenized["train"].column_names,
    )
    lm_dataset_cache[block_size] = lm_datasets
    return lm_datasets


def collate(batch):
    input_ids = torch.tensor([x["input_ids"] for x in batch], dtype=torch.long)
    labels = torch.tensor([x["labels"] for x in batch], dtype=torch.long)
    return input_ids, labels


def build_dataloaders(block_size, batch_size):
    cache_key = (block_size, batch_size)
    if cache_key in dataloader_cache:
        return dataloader_cache[cache_key]

    lm_datasets = build_lm_dataset(block_size)

    train_loader = torch.utils.data.DataLoader(
        lm_datasets["train"],
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate,
    )

    val_loader = torch.utils.data.DataLoader(
        lm_datasets["validation"],
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate,
    )

    dataloader_cache[cache_key] = (train_loader, val_loader)
    return train_loader, val_loader

# Attention Components

### Standard Causal Self-Attention

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0

        self.n_head = config.n_head
        self.head_dim = config.n_embd // config.n_head

        self.qkv = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.proj = nn.Linear(config.n_embd, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)

        self.register_buffer(
            "mask",
            torch.tril(torch.ones(config.block_size, config.block_size))
            .unsqueeze(0).unsqueeze(0),
        )

    def forward(self, x):
        B, T, C = x.shape

        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = torch.softmax(att, dim=-1)
        att = self.dropout(att)

        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)

### Multi-Query / Grouped-Query Attention

In [ ]:
class MultiQueryAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        assert config.n_head % config.kv_heads == 0

        self.n_head = config.n_head
        self.kv_heads = config.kv_heads
        self.head_dim = config.n_embd // config.n_head
        self.repeat_factor = self.n_head // self.kv_heads

        self.q_proj = nn.Linear(config.n_embd, config.n_embd)
        self.k_proj = nn.Linear(config.n_embd, self.kv_heads * self.head_dim)
        self.v_proj = nn.Linear(config.n_embd, self.kv_heads * self.head_dim)
        self.proj = nn.Linear(config.n_embd, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)

        self.register_buffer(
            "mask",
            torch.tril(torch.ones(config.block_size, config.block_size))
            .unsqueeze(0).unsqueeze(0),
        )

    def forward(self, x):
        B, T, C = x.shape

        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.kv_heads, self.head_dim).transpose(1, 2)

        k = k.repeat_interleave(self.repeat_factor, dim=1)
        v = v.repeat_interleave(self.repeat_factor, dim=1)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = torch.softmax(att, dim=-1)
        att = self.dropout(att)

        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)

### Sliding Window / Local Attention

In [ ]:
class SlidingWindowAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0

        self.n_head = config.n_head
        self.head_dim = config.n_embd // config.n_head
        self.window_size = config.window_size

        self.qkv = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.proj = nn.Linear(config.n_embd, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)

        positions = torch.arange(config.block_size)
        distance = positions[:, None] - positions[None, :]
        local_causal_mask = (distance >= 0) & (distance < self.window_size)

        self.register_buffer(
            "mask",
            local_causal_mask.unsqueeze(0).unsqueeze(0),
        )

    def forward(self, x):
        B, T, C = x.shape

        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        att = att.masked_fill(~self.mask[:, :, :T, :T], float("-inf"))
        att = torch.softmax(att, dim=-1)
        att = self.dropout(att)

        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)

### Causal Linear Attention

In [ ]:
class LinearCausalAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0

        self.n_head = config.n_head
        self.head_dim = config.n_embd // config.n_head
        self.eps = config.linear_attention_eps

        self.qkv = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.proj = nn.Linear(config.n_embd, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)

    def feature_map(self, x):
        return F.elu(x) + 1.0

    def forward(self, x):
        B, T, C = x.shape

        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        q = self.feature_map(q)
        k = self.feature_map(k)

        # Streaming causal linear attention avoids materializing a T x T attention matrix.
        # This is intentionally simple/readable; it may be slower than specialized kernels.
        cumulative_k = torch.zeros(B, self.n_head, self.head_dim, device=x.device, dtype=x.dtype)
        cumulative_kv = torch.zeros(
            B,
            self.n_head,
            self.head_dim,
            self.head_dim,
            device=x.device,
            dtype=x.dtype,
        )

        outputs = []
        for t in range(T):
            kt = k[:, :, t, :]
            vt = v[:, :, t, :]
            qt = q[:, :, t, :]

            cumulative_k = cumulative_k + kt
            cumulative_kv = cumulative_kv + torch.einsum("bhd,bhe->bhde", kt, vt)

            denom = torch.einsum("bhd,bhd->bh", qt, cumulative_k).unsqueeze(-1)
            yt = torch.einsum("bhd,bhde->bhe", qt, cumulative_kv) / (denom + self.eps)
            outputs.append(yt)

        y = torch.stack(outputs, dim=2)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.dropout(y)
        return self.proj(y)

# Feedforward + Transformer Block

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd),
            nn.GELU(),
            nn.Linear(4 * config.n_embd, config.n_embd),
            nn.Dropout(config.dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, config, attention_cls):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.attn = attention_cls(config)
        self.ln2 = nn.LayerNorm(config.n_embd)
        self.ff = FeedForward(config)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

# Full Model

In [ ]:
class TransformerModel(nn.Module):
    def __init__(self, config, attention_cls):
        super().__init__()
        self.config = config

        self.token_emb = nn.Embedding(config.vocab_size, config.n_embd)
        self.pos_emb = nn.Parameter(torch.zeros(1, config.block_size, config.n_embd))

        self.blocks = nn.ModuleList([
            Block(config, attention_cls)
            for _ in range(config.n_layer)
        ])

        self.ln_f = nn.LayerNorm(config.n_embd)
        self.head = nn.Linear(config.n_embd, config.vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.config.block_size

        tok = self.token_emb(idx)
        pos = self.pos_emb[:, :T, :]
        x = tok + pos

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits[:, :-1, :].contiguous().view(-1, logits.size(-1)),
                targets[:, 1:].contiguous().view(-1),
            )

        return logits, loss

# Training + Evaluation Helpers

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def sync_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def reset_peak_memory():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()


def get_peak_memory_mb():
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / (1024 ** 2)
    return 0.0


def evaluate(model, val_loader):
    model.eval()
    losses = []

    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            _, loss = model(x, y)
            losses.append(loss.item())

    model.train()
    return sum(losses) / len(losses)


def measure_inference_throughput(model, val_loader, max_batches=20):
    model.eval()
    total_tokens = 0

    sync_cuda()
    start_time = time.perf_counter()

    with torch.no_grad():
        for batch_idx, (x, _) in enumerate(val_loader):
            if batch_idx >= max_batches:
                break
            x = x.to(device)
            model(x)
            total_tokens += x.numel()

    sync_cuda()
    elapsed = time.perf_counter() - start_time
    model.train()

    return total_tokens / max(elapsed, 1e-9)


def save_checkpoint(path, model, optimizer, epoch, metrics):
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "metrics": metrics,
        },
        path,
    )

# Experiment Runner

In [ ]:
def train_experiment(
    attention_name,
    attention_cls,
    context_length,
    batch_size,
    epochs=EXPERIMENT_EPOCHS,
    seed=SEED,
):
    set_seed(seed)
    config = make_config(context_length)
    train_loader, val_loader = build_dataloaders(context_length, batch_size)

    model = TransformerModel(config, attention_cls).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)

    run_id = f"{attention_name}_ctx{context_length}_seed{seed}"
    metrics_path = os.path.join(LOG_DIR, f"{run_id}_metrics.csv")
    checkpoint_path = os.path.join(LOG_DIR, f"{run_id}_latest.pt")

    run_metrics = []

    for epoch in range(epochs):
        model.train()
        reset_peak_memory()
        pbar = tqdm(train_loader, desc=f"{attention_name} ctx={context_length} epoch={epoch}")

        total_train_loss = 0.0
        total_tokens = 0

        sync_cuda()
        start_time = time.perf_counter()

        for x, y in pbar:
            x, y = x.to(device), y.to(device)

            _, loss = model(x, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            batch_tokens = y[:, 1:].numel()
            total_tokens += batch_tokens
            total_train_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        sync_cuda()
        epoch_time = time.perf_counter() - start_time

        train_loss = total_train_loss / len(train_loader)
        train_throughput = total_tokens / epoch_time
        val_loss = evaluate(model, val_loader)
        val_perplexity = math.exp(min(val_loss, 20.0))
        inference_tokens_per_sec = measure_inference_throughput(model, val_loader)
        peak_gpu_memory_mb = get_peak_memory_mb()

        epoch_metrics = {
            "attention": attention_name,
            "context_length": context_length,
            "batch_size": batch_size,
            "seed": seed,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_perplexity": val_perplexity,
            "epoch_time_sec": epoch_time,
            "train_throughput_tokens_per_sec": train_throughput,
            "inference_throughput_tokens_per_sec": inference_tokens_per_sec,
            "peak_gpu_memory_mb": peak_gpu_memory_mb,
        }

        run_metrics.append(epoch_metrics)

        pd.DataFrame(run_metrics).to_csv(metrics_path, index=False)
        save_checkpoint(checkpoint_path, model, optimizer, epoch, epoch_metrics)

        print(epoch_metrics)
        print("saved metrics:", metrics_path)
        print("saved checkpoint:", checkpoint_path)

    return pd.DataFrame(run_metrics)


def run_variant_sweep(
    attention_name,
    attention_cls,
    context_lengths=CONTEXT_LENGTHS,
    epochs=EXPERIMENT_EPOCHS,
    seed=SEED,
):
    all_results = []

    for context_length in context_lengths:
        batch_size = BATCH_SIZES[context_length]

        while batch_size >= 1:
            try:
                result = train_experiment(
                    attention_name=attention_name,
                    attention_cls=attention_cls,
                    context_length=context_length,
                    batch_size=batch_size,
                    epochs=epochs,
                    seed=seed,
                )
                all_results.append(result)
                break
            except RuntimeError as err:
                if "out of memory" not in str(err).lower() or batch_size == 1:
                    raise
                print(f"OOM at context={context_length}, batch={batch_size}. Retrying with batch={batch_size // 2}.")
                batch_size = batch_size // 2
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

    combined = pd.concat(all_results, ignore_index=True)
    combined_path = os.path.join(LOG_DIR, f"{attention_name}_combined_metrics.csv")
    combined.to_csv(combined_path, index=False)
    print("saved combined metrics:", combined_path)
    return combined

# Run: Standard Full Attention Reference

In [ ]:
# standard_results = run_variant_sweep(
#     attention_name="standard_full_attention",
#     attention_cls=CausalSelfAttention,
#     context_lengths=CONTEXT_LENGTHS,
#     epochs=EXPERIMENT_EPOCHS,
#     seed=SEED,
# )

# display(standard_results)

# Run: Multi-Query Attention

In [ ]:
# mqa_results = run_variant_sweep(
#     attention_name="multi_query_attention",
#     attention_cls=MultiQueryAttention,
#     context_lengths=CONTEXT_LENGTHS,
#     epochs=EXPERIMENT_EPOCHS,
#     seed=SEED,
# )

# display(mqa_results)

# Run: Sliding Window Attention

In [ ]:
# sliding_results = run_variant_sweep(
#     attention_name="sliding_window_attention",
#     attention_cls=SlidingWindowAttention,
#     context_lengths=CONTEXT_LENGTHS,
#     epochs=EXPERIMENT_EPOCHS,
#     seed=SEED,
# )

# display(sliding_results)

# Run: Linear Causal Attention

In [ ]:
linear_results = run_variant_sweep(
    attention_name="linear_causal_attention",
    attention_cls=LinearCausalAttention,
    context_lengths=CONTEXT_LENGTHS,
    epochs=EXPERIMENT_EPOCHS,
    seed=SEED,
)

display(linear_results)

# Aggregate Saved Results

In [ ]:
metric_files = [
    os.path.join(LOG_DIR, name)
    for name in os.listdir(LOG_DIR)
    if name.endswith("_metrics.csv") and "combined" not in name
]

if metric_files:
    all_metrics = pd.concat([pd.read_csv(path) for path in metric_files], ignore_index=True)
    display(all_metrics)

    final_epoch_metrics = all_metrics.sort_values("epoch").groupby(
        ["attention", "context_length", "seed"],
        as_index=False,
    ).tail(1)

    display(final_epoch_metrics.sort_values(["context_length", "val_perplexity"]))

    aggregate_path = os.path.join(LOG_DIR, "attention_variants_all_metrics.csv")
    all_metrics.to_csv(aggregate_path, index=False)
    print("saved aggregate metrics:", aggregate_path)
else:
    print("No metric files found yet in", LOG_DIR)